In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchsummary import summary
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
import utils

In [2]:
data = utils.load_data()

100%|██████████| 45/45 [01:09<00:00,  1.54s/it]


In [3]:
utils.indexes_to_phonemes(data[0][1]['seq_class_ids'][0][:13]), data[0][1]['sentence_label'][0]

(['W', 'IH', 'CH', ' | ', 'IH', 'Z', ' | ', 'M', 'OW', 'S', 'T', ' | ', 'AH'],
 'Which is most unfortunate because we all lose out.')

### Probabilistic Inputs (Soft Inputs)

1.  **Input Shape**: Instead of `[Batch, Length]` (indices), the input becomes `[Batch, Length, Num_Phonemes]` (probabilities).
2.  **Projection Layer**: Replace the `nn.Embedding` (lookup table) with a `nn.Linear` layer.
    *   Mathematically, looking up an embedding for index $i$ is the same as multiplying a one-hot vector of $i$ by a linear matrix.
    *   By using `nn.Linear`, we can multiply *any* probability vector (not just one-hot) by the matrix.

modes:
*   **Training**: You can pass indices (ground truth) -> It converts them to one-hot vectors internally.
*   **Inference**: You can pass the probability outputs from your GRU -> It projects them directly.

In [ ]:
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

class PhonemeToTextTransformer(nn.Module):
    def __init__(self, num_phonemes, num_text_tokens, d_model=256, nhead=4, num_encoder_layers=3, num_decoder_layers=3, dim_feedforward=1024, dropout=0.1):
        super(PhonemeToTextTransformer, self).__init__()
        self.d_model = d_model
        self.num_phonemes = num_phonemes
        
        # CHANGED: Use Linear instead of Embedding to support probability inputs
        # If input is indices, we one-hot encode them first.
        self.phoneme_projection = nn.Linear(num_phonemes, d_model)
        
        self.text_embedding = nn.Embedding(num_text_tokens, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout)
        
        # Transformer
        # batch_first=True is easier to work with
        self.transformer = nn.Transformer(d_model=d_model, nhead=nhead, 
                                          num_encoder_layers=num_encoder_layers, 
                                          num_decoder_layers=num_decoder_layers, 
                                          dim_feedforward=dim_feedforward, 
                                          dropout=dropout, batch_first=True)
        
        # Output Head
        self.fc_out = nn.Linear(d_model, num_text_tokens)
        
    def forward(self, src, tgt, src_key_padding_mask=None, tgt_key_padding_mask=None, memory_key_padding_mask=None):
        '''
        src: [batch_size, src_len] (Indices) OR [batch_size, src_len, num_phonemes] (Probabilities)
        tgt: [batch_size, tgt_len] (Text token indices)
        '''
        
        # Handle Input Type
        if src.dim() == 2:
            # If indices, convert to One-Hot (Hard Probabilities)
            # src: [batch, len] -> [batch, len, num_phonemes]
            src = torch.nn.functional.one_hot(src, num_classes=self.num_phonemes).float()
        
        # Generate mask to prevent decoder from looking ahead
        tgt_mask = self.transformer.generate_square_subsequent_mask(tgt.size(1)).to(tgt.device)
        
        # Embed + Positional Encoding
        # Project probabilities to d_model
        src_emb = self.phoneme_projection(src) * math.sqrt(self.d_model)
        src_emb = self.pos_encoder(src_emb)
        
        tgt_emb = self.pos_encoder(self.text_embedding(tgt) * math.sqrt(self.d_model))
        
        # Transformer Pass
        outs = self.transformer(src_emb, tgt_emb, tgt_mask=tgt_mask, 
                                src_key_padding_mask=src_key_padding_mask, 
                                tgt_key_padding_mask=tgt_key_padding_mask, 
                                memory_key_padding_mask=memory_key_padding_mask)
        
        # Project to vocabulary size
        return self.fc_out(outs)

# Example Instantiation
# Assuming 41 phonemes and a simple character-level text vocabulary of ~500 characters
p2t_model = PhonemeToTextTransformer(
    num_phonemes=41, 
    num_text_tokens=500, 
    d_model=128, 
    nhead=4, 
    num_encoder_layers=2, 
    num_decoder_layers=2
)

print(p2t_model)


## Training transformer

1.  **Tokenize the Text**: Convert English sentences into sequences of integers (indices).
2.  **Prepare the Data**: Create a PyTorch `Dataset` and `DataLoader` to batch the pairs of `(Phoneme Sequence, Text Sequence)`.
3.  **Run the Training Loop**: Optimize the model to predict the next text token given the phonemes and previous text tokens.

### 1. Text Tokenizer
Character-level tokenizer.

In [ ]:
class CharTokenizer:
    def __init__(self):
        # 0: Pad, 1: Start of Sentence (SOS), 2: End of Sentence (EOS)
        self.char2idx = {'<pad>': 0, '<sos>': 1, '<eos>': 2}
        self.idx2char = {0: '<pad>', 1: '<sos>', 2: '<eos>'}
        self.vocab_size = 3

    def fit(self, sentences):
        unique_chars = set("".join(sentences))
        for char in sorted(unique_chars):
            if char not in self.char2idx:
                self.char2idx[char] = self.vocab_size
                self.idx2char[self.vocab_size] = char
                self.vocab_size += 1
    
    def encode(self, sentence):
        # Add SOS at start and EOS at end
        return [self.char2idx['<sos>']] + [self.char2idx[c] for c in sentence] + [self.char2idx['<eos>']]

    def decode(self, indices):
        # Convert back to string, ignoring special tokens
        return "".join([self.idx2char[idx] for idx in indices if idx not in [0, 1, 2]])

# Collect all sentences to build vocabulary
all_sentences = []
for session_data in data:
    if len(session_data) > 1: # Ensure train set exists
        # Handle bytes vs str
        sentences = [s.decode('utf-8') if isinstance(s, bytes) else s for s in session_data[1]['sentence_label']]
        all_sentences.extend(sentences)

# Initialize and fit tokenizer
tokenizer = CharTokenizer()
tokenizer.fit(all_sentences)
print(f"Vocabulary Size: {tokenizer.vocab_size}")
print(f"Example encoding: 'hello' -> {tokenizer.encode('hello')}")

### 2. Dataset and DataLoader
We create a custom Dataset to handle the pairing of phonemes and text.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class PhonemeTextDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.samples = []
        self.tokenizer = tokenizer
        
        # Flatten the data structure
        for session in data:
            if len(session) < 2: continue
            train_data = session[1] # 1 is train
            
            for i in range(len(train_data['seq_class_ids'])):
                phonemes = train_data['seq_class_ids'][i]
                sentence = train_data['sentence_label'][i]
                
                if phonemes is None or sentence is None: continue
                
                # Clean sentence
                if isinstance(sentence, bytes): 
                    sentence = sentence.decode('utf-8')
                
                self.samples.append((phonemes, sentence))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        phonemes, sentence = self.samples[idx]
        
        # Convert to tensors
        # Phonemes: already indices. 
        # Note: If your phonemes have a 'blank' token 0, it might conflict with padding 0. 
        # Ideally, ensure padding is a unique index. Here we assume 0 is safe or handled.
        src = torch.tensor(phonemes, dtype=torch.long)
        
        # Text: Encode to indices
        tgt = torch.tensor(self.tokenizer.encode(sentence), dtype=torch.long)
        
        return src, tgt

def collate_fn(batch):
    # Pad sequences to max length in batch
    src_batch, tgt_batch = zip(*batch)
    
    # Pad with 0 (assuming 0 is <pad> for both, or adjust accordingly)
    src_padded = pad_sequence(src_batch, batch_first=True, padding_value=0) 
    tgt_padded = pad_sequence(tgt_batch, batch_first=True, padding_value=0) 
    
    return src_padded, tgt_padded

# Create Dataset and DataLoader
dataset = PhonemeTextDataset(data, tokenizer)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)

print(f"Dataset size: {len(dataset)}")
first_batch = next(iter(dataloader))
print(f"Batch shapes - Src: {first_batch[0].shape}, Tgt: {first_batch[1].shape}")

### 3. Training Loop
Now we train the model.
*   **Input to Decoder (`tgt_input`)**: The target sequence *excluding* the last token (e.g., `<sos> h e l l o`).
*   **Target for Loss (`tgt_output`)**: The target sequence *excluding* the first token (e.g., `h e l l o <eos>`).
*   **Teacher Forcing**: We feed the correct previous tokens to the decoder during training.

In [ ]:
# --- Hyperparameters ---
EPOCHS = 5
LR = 0.0005
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Model Initialization ---
# Re-initialize model with correct vocab size
transformer_model = PhonemeToTextTransformer(
    num_phonemes=41, 
    num_text_tokens=tokenizer.vocab_size,
    d_model=64,
    nhead=2,
    num_encoder_layers=2,
    num_decoder_layers=2,
    dropout=0.1
).to(DEVICE)

# --- Optimizer & Loss ---
optimizer = optim.AdamW(transformer_model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss(ignore_index=0) # Ignore padding in loss calculation

# --- Training Loop ---
print(f"Starting training on {DEVICE}...")

os.makedirs("transformer_model_weights", exist_ok=True)

transformer_model.train()

for epoch in range(EPOCHS):
    total_loss = 0
    
    for batch_idx, (src, tgt) in enumerate(dataloader):
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        
        # Prepare inputs and targets
        # tgt_input: <sos> ... n-1
        tgt_input = tgt[:, :-1]
        
        # tgt_output: 1 ... <eos>
        tgt_output = tgt[:, 1:]
        
        # Create Padding Masks (Optional but recommended for variable lengths)
        # src_padding_mask = (src == 0)
        # tgt_padding_mask = (tgt_input == 0)
        
        optimizer.zero_grad()
        
        # Forward Pass
        # The model handles the look-ahead mask internally
        output = transformer_model(src, tgt_input)
        
        # Reshape for Loss
        # Output: [batch, seq_len, vocab_size] -> [batch * seq_len, vocab_size]
        # Target: [batch, seq_len] -> [batch * seq_len]
        output_flat = output.reshape(-1, output.shape[-1])
        tgt_flat = tgt_output.reshape(-1)
        
        loss = criterion(output_flat, tgt_flat)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(transformer_model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        
        if batch_idx % 1 == 0:
            print(f"Epoch {epoch+1} | Batch {batch_idx}/{len(dataloader)} | Loss: {loss.item():.4f}", end='\r')
            
    avg_loss = total_loss / len(dataloader)

    torch.save(transformer_model.state_dict(), f"transformer_model_weights/model_epoch_{epoch+1}.pth")
    print(f"\nEpoch {epoch+1} Complete | Avg Loss: {avg_loss:.4f}")

print("Training Finished!")

## Training with Simulated GRU Outputs (Robustness)

To train the Transformer to handle the **uncertainty** and **errors** of the GRU, we shouldn't just train on perfect ground truth phonemes.

Instead, we will **simulate** the GRU's output during training by:
1.  **Hard Errors**: Randomly changing some phonemes (simulating a wrong prediction).
2.  **Soft Uncertainty**: Converting the hard index to a probability distribution (e.g., 90% correct, 10% noise).

This forces the Transformer to learn to "fix" the input using context.

In [ ]:
def simulate_gru_output(indices, num_phonemes, error_rate=0.15, confidence=0.9):
    """
    Simulates the noisy, probabilistic output of a GRU.
    1. Hard Errors: Randomly changes some phonemes to wrong ones.
    2. Uncertainty: Converts to probabilities with < 1.0 confidence.
    """
    batch_size, seq_len = indices.shape
    device = indices.device
    
    # 1. Simulate Hard Errors (Substitution)
    # Create a mask of errors
    error_mask = torch.rand(indices.shape, device=device) < error_rate
    # Generate random phonemes
    random_phonemes = torch.randint(0, num_phonemes, indices.shape, device=device)
    # Apply errors (avoid modifying padding 0 if possible)
    # Assuming 0 is PAD, we don't want to turn real data into PAD or vice versa usually, 
    # but for simplicity we just avoid touching 0 indices.
    mask_non_pad = (indices != 0)
    noisy_indices = torch.where(error_mask & mask_non_pad, random_phonemes, indices)
    
    # 2. Convert to Soft Probabilities (Label Smoothing)
    # Start with One-Hot
    one_hot = torch.nn.functional.one_hot(noisy_indices, num_classes=num_phonemes).float()
    
    # Apply smoothing: Peak = confidence, Others = (1-confidence)/(N-1)
    # Simplified: Mix with uniform distribution
    uniform = torch.ones_like(one_hot) / num_phonemes
    soft_output = confidence * one_hot + (1 - confidence) * uniform
    
    # 3. Add random noise to distribution (Jitter)
    noise = torch.randn_like(soft_output) * 0.05
    soft_output = soft_output + noise
    
    # Re-normalize to sum to 1
    soft_output = torch.softmax(soft_output, dim=-1)
    
    # Restore padding to be clean (optional, but helps model focus)
    # If index was 0, make it [1, 0, 0...]
    pad_mask = (indices == 0)
    pad_one_hot = torch.zeros_like(soft_output)
    pad_one_hot[..., 0] = 1.0
    soft_output = torch.where(pad_mask.unsqueeze(-1), pad_one_hot, soft_output)
    
    return soft_output

# Test the simulation
sample_indices = torch.tensor([[1, 5, 10, 0]]) # Example batch
soft_input = simulate_gru_output(sample_indices, num_phonemes=41)
print(f"Original: {sample_indices}")
print(f"Soft Shape: {soft_input.shape}")
print(f"Soft Input (First Token):\n{soft_input[0, 0]}")

In [ ]:
# --- Advanced Training Loop with Soft Inputs ---
print(f"Starting Robust Training on {DEVICE}...")

# Re-init model to start fresh
transformer_model = PhonemeToTextTransformer(
    num_phonemes=41, 
    num_text_tokens=tokenizer.vocab_size,
    d_model=64,
    nhead=2,
    num_encoder_layers=2,
    num_decoder_layers=2,
    dropout=0.1
).to(DEVICE)

optimizer = optim.AdamW(transformer_model.parameters(), lr=LR)

for epoch in range(EPOCHS):
    total_loss = 0
    transformer_model.train()
    
    for batch_idx, (src, tgt) in enumerate(dataloader):
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        
        # --- SIMULATE GRU OUTPUT ---
        # Convert hard indices to soft, noisy probabilities
        src_soft = simulate_gru_output(src, num_phonemes=41, error_rate=0.15, confidence=0.8)
        
        # Prepare targets
        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]
        
        optimizer.zero_grad()
        
        # Pass SOFT inputs to the model
        output = transformer_model(src_soft, tgt_input)
        
        output_flat = output.reshape(-1, output.shape[-1])
        tgt_flat = tgt_output.reshape(-1)
        
        loss = criterion(output_flat, tgt_flat)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(transformer_model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        
        if batch_idx % 10 == 0:
            print(f"Epoch {epoch+1} | Batch {batch_idx}/{len(dataloader)} | Loss: {loss.item():.4f}", end='\r')
            
    avg_loss = total_loss / len(dataloader)
    print(f"\nEpoch {epoch+1} Complete | Avg Loss: {avg_loss:.4f}")

print("Robust Training Finished!")